In [29]:
import numpy as np
import torch
import torch.nn as nn
import time
import matplotlib.pyplot as plt
import os
import logging
from datetime import datetime
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm  # Use notebook-friendly tqdm

# Make sure your custom modules are available
from model import DeepONet
from utils import TrajectoryDataset, load_multi_traj_data, run_model_visualization

# Helper function from your script
def ellip_vol(model):
    d = model.V.log_diag_L.numel()
    c_val = model.c ** 2
    
    # Compute det(Q)^(-1/2)
    log_det_Q = 2 * torch.sum(model.V.log_diag_L)
    det_factor = torch.exp(-0.5 * log_det_Q)

    # Final volume
    if model.trainable_c:
        vol = (c_val**(d/2)) * det_factor
    else:
        vol = det_factor
    return vol

# --- Device Configuration ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(0)
np.random.seed(0)

print(f"Using device: {device}")

Using device: cpu


In [30]:
# model_folder = 'Trained_Models/0818_15/E20000_TS0.05_branchConv0_trunkHidden3__proj_LamRegVol1.0_C01.0_diagQ_dim256_printgrad'

params = {
    'epochs': 20000,
    'bsize': 2048,
    'lam_reg_vol': 1.0,
    'project': True,
    'tag': '',
    'c_init': 1.0,
    'trainable_c': True,
    'trunk_scale': 0.05,
    'diag_Q': True,
    'output_dim': 256,
    'branch_conv_channels': [],
    'branch_fc_dims': [256],
    'trunk_hidden_dims': [256, 256, 256]
}

# # --- Setup Directories ---
# now = datetime.now()
# save_time_str = now.strftime("%m%d_%H")
# reg_name = ''
# if params['trainable_c']: reg_name += 'cTrain'
# if params['project']: reg_name += f'_proj_LamRegVol{params["lam_reg_vol"]}_C0{params["c_init"]}'
# if params['diag_Q']: reg_name += '_diagQ'
    
# save_name = f'E{params["epochs"]}_TS{params["trunk_scale"]}_branchConv{len(params["branch_conv_channels"])}_trunkHidden{len(params["trunk_hidden_dims"])}_{reg_name}_{params["tag"]}'
# save_dir = os.path.join('Trained_Models', save_time_str, save_name)

# params['save_dir'] = save_dir
# model_folder = save_dir
# figs_folder = os.path.join(save_dir, 'eval_results')

# os.makedirs(model_folder, exist_ok=True)
# os.makedirs(figs_folder, exist_ok=True)

# print(f"Results will be saved in: {save_dir}")

In [33]:
file_dir = '../../../Data/KS_data_batched_l100.53_grid512_M8_T200.0_dt0.005_dt_sample0.2_amp20.0/data.npz'
data = np.load(file_dir, allow_pickle=True)

train_dataset, val_dataset = load_multi_traj_data(data, params['trunk_scale'])

train_loader = DataLoader(train_dataset, batch_size=params['bsize'], shuffle=True, pin_memory=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=params['bsize'], shuffle=False)

print(f"Created DataLoaders with {len(train_dataset)} training samples and {len(val_dataset)} validation samples.")

Created DataLoaders with 5994 training samples and 1998 validation samples.


In [34]:
# Assuming u_batch is of shape (num_traj, traj_length, traj_dim)
m = s = data['u_batch'].shape[2]
n = 1

model_params = {
    'm': m,
    'n': n,
    'trainable_c': params['trainable_c'],
    'c0': params['c_init'],
    'project': params['project'],
    'diag_Q': params['diag_Q'],
    'branch_conv_channels': params['branch_conv_channels'],
    'branch_fc_dims': params['branch_fc_dims'],
    'trunk_hidden_dims': params['trunk_hidden_dims'],
    'output_dim': params['output_dim']
}

model = DeepONet(model_params).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
loss_func = torch.nn.MSELoss()

num_params = sum(v.numel() for v in model.parameters() if v.requires_grad)
print(f'Model initialized with {num_params:,} trainable parameters.')

Auto-detected flattened size for FC layer: 512
--- Initialized Branch Net Structure ---
Branch(
  (activation): ReLU()
  (conv_net): Sequential()
  (fc_net): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
  )
)

--- Initialized Trunk Net Structure ---
Trunk(
  (activation): ReLU()
  (net): Sequential(
    (0): Linear(in_features=1, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=256, bias=True)
    (5): ReLU()
    (6): Linear(in_features=256, out_features=256, bias=True)
  )
)
----------------------------------------
Projection layer included
V_elliptical initialized with a DIAGONAL Q.
Model initialized with 526,850 trainable parameters.


In [35]:
model.load_state_dict(torch.load('./model_epoch_best.pt', map_location=device))

<All keys matched successfully>

In [42]:
model.eval()
print(torch.min(model.V.log_diag_L))
# model.c
torch.exp(torch.sum(model.V.log_diag_L))

tensor(-0.5578, grad_fn=<MinBackward1>)


tensor(5.9724e+08, grad_fn=<ExpBackward0>)

In [28]:

params = np.load('./model_params.npz')
m = 512
n = 1
model_params = {
    'm': m,
    'n': n,
    'trainable_c': params['trainable_c'],
    'c0': 1.0, #params['c_init'],
    'project': params['project'],
    'diag_Q': params['diag_Q'],
    'branch_conv_channels': params['branch_conv_channels'],
    'branch_fc_dims': params['branch_fc_dims'],
    'trunk_hidden_dims': params['trunk_hidden_dims'],
    'output_dim': params['output_dim']
}

print(model_params['trunk_hidden_dims'])

model = DeepONet(model_params)

# model_folder = 'Trained_Models/0818_15/E20000_TS0.05_branchConv0_trunkHidden3__proj_LamRegVol1.0_C01.0_diagQ_dim256_printgrad'
model.load_state_dict(torch.load(f"./model_epoch_best.pt",map_location=torch.device('cpu')))

[256 256 256]
Auto-detected flattened size for FC layer: 512
--- Initialized Branch Net Structure ---
Branch(
  (activation): ReLU()
  (conv_net): Sequential()
  (fc_net): Sequential()
)

--- Initialized Trunk Net Structure ---
Trunk(
  (activation): ReLU()
  (net): Sequential(
    (0): Linear(in_features=513, out_features=513, bias=True)
    (1): ReLU()
    (2): Linear(in_features=513, out_features=513, bias=True)
  )
)
----------------------------------------
Projection layer included
V_elliptical initialized with a DIAGONAL Q.


RuntimeError: Error(s) in loading state_dict for DeepONet:
	Unexpected key(s) in state_dict: "Branch.fc_net.0.weight", "Branch.fc_net.0.bias", "Branch.fc_net.2.weight", "Branch.fc_net.2.bias", "Trunk.net.4.weight", "Trunk.net.4.bias", "Trunk.net.6.weight", "Trunk.net.6.bias". 
	size mismatch for Trunk.net.0.weight: copying a param with shape torch.Size([256, 1]) from checkpoint, the shape in current model is torch.Size([513, 513]).
	size mismatch for Trunk.net.0.bias: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([513]).
	size mismatch for Trunk.net.2.weight: copying a param with shape torch.Size([256, 256]) from checkpoint, the shape in current model is torch.Size([513, 513]).
	size mismatch for Trunk.net.2.bias: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([513]).